# 15 — P2: MILP Encoding of RMSNorm

**Plan 2 — Phase 4.2.** Sound MILP encoding for one RMSNorm layer:
$$y_i = \gamma_i \cdot \frac{x_i}{\sqrt{\tfrac{1}{D}\sum_j x_j^2 + \varepsilon}}$$

The encoding composes three operations, each with provable soundness:

1. **Per-coordinate `x_i²`** — convex PWL bracket (Phase 4.1, `pwl_square`) on
   `[x_lo[i], x_up[i]]`.
2. **Average `mean_xsq = (1/D) Σ xsq[i]`** — exact linear constraint.
3. **`inv_rms = 1/√(mean_xsq + ε)`** — convex PWL bracket on
   `[mean_xsq_lo+ε, mean_xsq_up+ε]`.
4. **Element-wise `y_i = γ_i · x_i · inv_rms`** — McCormick envelope for the
   bilinear product `x_i · inv_rms`, then scaled by the constant `γ_i`.

McCormick for `z = a·b` over `a ∈ [a_l, a_u]`, `b ∈ [b_l, b_u]` (both intervals):
```
z ≥ a_l·b + a·b_l − a_l·b_l
z ≥ a_u·b + a·b_u − a_u·b_u
z ≤ a_u·b + a·b_l − a_u·b_l
z ≤ a_l·b + a·b_u − a_l·b_u
```

## Soundness contract
For an input box `x ∈ [x_lo, x_up]`, the encoding produces `y_lo, y_up` such
that for **every** point `x* ∈ [x_lo, x_up]`, the true RMSNorm output
`y*_i = γ_i · x*_i / √(mean(x*²) + ε)` satisfies `y_lo[i] ≤ y*_i ≤ y_up[i]`.

## Tests in this notebook
1. Build the encoding for several `(D, x-box)` configurations.
2. For each output coordinate, solve `min y_i` and `max y_i` with the input
   box as the only constraint on `x`. This gives a sound MILP-certified
   interval `[y_min[i], y_max[i]]`.
3. Sample `K` interior points `x_pin ∈ [x_lo, x_up]`. For each, compute the
   true PyTorch RMSNorm output and assert `y_min[i] ≤ true_y[i] ≤ y_max[i]`.
4. Report max bracket violation (must be ≤ 1e-6) and mean width.


In [1]:
!pip install -q numpy torch gurobipy

In [2]:
from __future__ import annotations
import math, json, time, warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, List, Tuple

import numpy as np
import torch
import torch.nn as nn

try:
    import gurobipy as gp
    from gurobipy import GRB
    HAS_GUROBI = True
except Exception as e:
    HAS_GUROBI = False
    print(f'Gurobi not available ({e}); MILP tests will be skipped.')

warnings.filterwarnings('ignore')
rng = np.random.default_rng(1234)
torch.manual_seed(1234)
print(f'NumPy {np.__version__}  Torch {torch.__version__}  Gurobi={HAS_GUROBI}')

NumPy 2.0.2  Torch 2.10.0+cu128  Gurobi=True


In [3]:
# ── PWL bracket primitives (verbatim from notebook 14) ───────────────────
@dataclass
class PWLBracket:
    name: str
    breakpoints: List[float]
    slope_lo: List[float]
    int_lo:   List[float]
    slope_up: List[float]
    int_up:   List[float]

    @property
    def n_pieces(self) -> int:
        return len(self.breakpoints) - 1

    @property
    def domain(self) -> Tuple[float, float]:
        return (self.breakpoints[0], self.breakpoints[-1])

    def evaluate(self, x: float) -> Tuple[float, float]:
        a, b = self.domain
        x_c = max(a, min(b, x))
        for k in range(self.n_pieces):
            if self.breakpoints[k] - 1e-12 <= x_c <= self.breakpoints[k+1] + 1e-12:
                lo = self.slope_lo[k] * x_c + self.int_lo[k]
                up = self.slope_up[k] * x_c + self.int_up[k]
                return float(lo), float(up)
        raise RuntimeError(f'x={x} not in segments of {self.name}')


def _line_through(p1, p2):
    x1, y1 = p1; x2, y2 = p2
    s = (y2 - y1) / (x2 - x1); intc = y1 - s * x1
    return s, intc

def _tangent_at(f, df, x0):
    s = df(x0); intc = f(x0) - s * x0
    return s, intc

def build_pwl_convex(f, df, a, b, n_pieces, name):
    bps = list(np.linspace(a, b, n_pieces + 1))
    slope_up, int_up, slope_lo, int_lo = [], [], [], []
    for k in range(n_pieces):
        x_l, x_r = bps[k], bps[k+1]
        s_u, i_u = _line_through((x_l, f(x_l)), (x_r, f(x_r)))   # secant (upper)
        slope_up.append(s_u); int_up.append(i_u)
        x_m = 0.5 * (x_l + x_r)
        s_l, i_l = _tangent_at(f, df, x_m)                        # tangent (lower)
        slope_lo.append(s_l); int_lo.append(i_l)
    return PWLBracket(name, bps, slope_lo, int_lo, slope_up, int_up)

def build_pwl_concave(f, df, a, b, n_pieces, name):
    bps = list(np.linspace(a, b, n_pieces + 1))
    slope_up, int_up, slope_lo, int_lo = [], [], [], []
    for k in range(n_pieces):
        x_l, x_r = bps[k], bps[k+1]
        s_l, i_l = _line_through((x_l, f(x_l)), (x_r, f(x_r)))    # secant (lower)
        slope_lo.append(s_l); int_lo.append(i_l)
        x_m = 0.5 * (x_l + x_r)
        s_u, i_u = _tangent_at(f, df, x_m)                         # tangent (upper)
        slope_up.append(s_u); int_up.append(i_u)
    return PWLBracket(name, bps, slope_lo, int_lo, slope_up, int_up)

def pwl_square(a, b, n_pieces=8):
    return build_pwl_convex(lambda x: x*x, lambda x: 2*x, a, b, n_pieces, 'square')

def pwl_inv_sqrt_pos(a, b, n_pieces=8):
    """1/sqrt(t) on [a,b] with a > 0 — convex."""
    assert a > 0, 'pwl_inv_sqrt_pos requires a > 0'
    return build_pwl_convex(lambda t: 1.0/math.sqrt(t),
                            lambda t: -0.5 * t**(-1.5),
                            a, b, n_pieces, 'inv_sqrt')

In [4]:
# ── Big-M PWL bracket encoder (verbatim from notebook 14) ────────────────
def add_pwl_bracket(model, x_var, y_var, bracket: PWLBracket,
                    big_M: float | None = None, prefix: str = ''):
    if not HAS_GUROBI:
        raise RuntimeError('Gurobi not available')
    n = bracket.n_pieces; bps = bracket.breakpoints
    a, b = bps[0], bps[-1]
    if big_M is None:
        scope = max(
            max(abs(bracket.slope_lo[k]) * (b - a) + abs(bracket.int_lo[k]) for k in range(n)),
            max(abs(bracket.slope_up[k]) * (b - a) + abs(bracket.int_up[k]) for k in range(n)),
        )
        big_M = max(2.0 * scope, 1.0)
    delta = [model.addVar(vtype=GRB.BINARY, name=f'{prefix}delta_{k}') for k in range(n)]
    model.addConstr(gp.quicksum(delta) == 1, name=f'{prefix}delta_sum')
    for k in range(n):
        x_l, x_r = bps[k], bps[k+1]
        model.addConstr(x_var >= x_l - big_M * (1 - delta[k]), name=f'{prefix}seg{k}_xlo')
        model.addConstr(x_var <= x_r + big_M * (1 - delta[k]), name=f'{prefix}seg{k}_xup')
        s_lo, i_lo = bracket.slope_lo[k], bracket.int_lo[k]
        s_up, i_up = bracket.slope_up[k], bracket.int_up[k]
        model.addConstr(y_var >= s_lo * x_var + i_lo - big_M * (1 - delta[k]),
                        name=f'{prefix}seg{k}_ylo')
        model.addConstr(y_var <= s_up * x_var + i_up + big_M * (1 - delta[k]),
                        name=f'{prefix}seg{k}_yup')
    return delta

In [5]:
# ── McCormick envelope for z = a · b ─────────────────────────────────────
def add_mccormick_bilinear(model, a_var, b_var, a_l, a_u, b_l, b_u,
                           prefix: str = ''):
    """
    Add z and 4 linear constraints encoding the McCormick envelope of z = a·b.
    Bounds [a_l, a_u] and [b_l, b_u] must contain the realised values of
    a_var and b_var; if a_var is a constant, pass it as a_l == a_u.

    Returns the new Gurobi var z (real, free).
    """
    if not HAS_GUROBI:
        raise RuntimeError('Gurobi not available')
    z_lo = min(a_l*b_l, a_l*b_u, a_u*b_l, a_u*b_u)
    z_up = max(a_l*b_l, a_l*b_u, a_u*b_l, a_u*b_u)
    z = model.addVar(lb=z_lo, ub=z_up, name=f'{prefix}z')
    # z >= a_l·b + b_l·a - a_l·b_l   (lower-left corner)
    model.addConstr(z >= a_l*b_var + b_l*a_var - a_l*b_l, name=f'{prefix}mc1')
    # z >= a_u·b + b_u·a - a_u·b_u   (upper-right corner)
    model.addConstr(z >= a_u*b_var + b_u*a_var - a_u*b_u, name=f'{prefix}mc2')
    # z <= a_u·b + b_l·a - a_u·b_l   (cross 1)
    model.addConstr(z <= a_u*b_var + b_l*a_var - a_u*b_l, name=f'{prefix}mc3')
    # z <= a_l·b + b_u·a - a_l·b_u   (cross 2)
    model.addConstr(z <= a_l*b_var + b_u*a_var - a_l*b_u, name=f'{prefix}mc4')
    return z

In [6]:
# ── MILP encoding of RMSNorm ─────────────────────────────────────────────
def encode_rmsnorm(milp, x_vars, x_lo, x_up, gamma, eps_rms,
                   n_pieces: int = 8, prefix: str = ''):
    """
    Add MILP variables/constraints encoding y_i = γ_i · x_i / √(mean(x²) + ε).

    Inputs
    ------
    milp     : Gurobi Model
    x_vars   : list of D Gurobi vars representing the input vector x
    x_lo,x_up: 1-D arrays of length D — sound bounds on each x_i
    gamma    : 1-D array of length D — RMSNorm scale parameter (constants)
    eps_rms  : float — RMSNorm epsilon

    Returns
    -------
    y_vars   : list of D Gurobi vars representing y
    info     : dict with derived intermediate bounds (for inspection)
    """
    D = len(x_vars)
    x_lo = np.asarray(x_lo, dtype=float)
    x_up = np.asarray(x_up, dtype=float)
    gamma = np.asarray(gamma, dtype=float)

    # ── 1. xsq[i] = x[i]^2 via PWL on [x_lo[i], x_up[i]] ──
    xsq_vars = []
    xsq_lo_arr = np.zeros(D); xsq_up_arr = np.zeros(D)
    for i in range(D):
        a, b = float(x_lo[i]), float(x_up[i])
        if a == b:
            # degenerate: x_i is fixed; xsq is a constant
            xsq_lo_arr[i] = xsq_up_arr[i] = a*a
            v = milp.addVar(lb=a*a, ub=a*a, name=f'{prefix}xsq_{i}')
            milp.addConstr(v == a*a, name=f'{prefix}xsq_{i}_pin')
            xsq_vars.append(v); continue
        bracket = pwl_square(a, b, n_pieces=n_pieces)
        # Sound IBP for x²:
        if a <= 0 <= b:
            xsq_lo_arr[i] = 0.0
        else:
            xsq_lo_arr[i] = min(a*a, b*b)
        xsq_up_arr[i] = max(a*a, b*b)
        v = milp.addVar(lb=xsq_lo_arr[i] - 1e-3, ub=xsq_up_arr[i] + 1e-3,
                        name=f'{prefix}xsq_{i}')
        add_pwl_bracket(milp, x_vars[i], v, bracket, prefix=f'{prefix}xsq_{i}_')
        xsq_vars.append(v)

    # ── 2. mean_xsq = (1/D) Σ xsq_i — exact linear ──
    mean_xsq_lo = float(xsq_lo_arr.mean())
    mean_xsq_up = float(xsq_up_arr.mean())
    mean_xsq = milp.addVar(lb=mean_xsq_lo, ub=mean_xsq_up,
                           name=f'{prefix}mean_xsq')
    milp.addConstr(mean_xsq == gp.quicksum(xsq_vars) / D,
                   name=f'{prefix}mean_xsq_def')

    # ── 3. denom = mean_xsq + eps_rms (offset, linear) ──
    denom_lo = mean_xsq_lo + eps_rms
    denom_up = mean_xsq_up + eps_rms
    denom_var = milp.addVar(lb=denom_lo, ub=denom_up, name=f'{prefix}denom')
    milp.addConstr(denom_var == mean_xsq + eps_rms, name=f'{prefix}denom_def')

    # ── 4. inv_rms = 1/sqrt(denom) via PWL ──
    if denom_lo <= 0:
        raise ValueError(f'denom_lo={denom_lo} ≤ 0; eps_rms must be > 0')
    inv_sqrt_bracket = pwl_inv_sqrt_pos(denom_lo, denom_up, n_pieces=n_pieces)
    inv_rms_lo = 1.0 / math.sqrt(denom_up)
    inv_rms_up = 1.0 / math.sqrt(denom_lo)
    inv_rms = milp.addVar(lb=inv_rms_lo - 1e-3, ub=inv_rms_up + 1e-3,
                          name=f'{prefix}inv_rms')
    add_pwl_bracket(milp, denom_var, inv_rms, inv_sqrt_bracket,
                    prefix=f'{prefix}inv_rms_')

    # ── 5. y_i = γ_i · (x_i · inv_rms) via McCormick + linear scaling ──
    y_vars = []
    for i in range(D):
        a_l, a_u = float(x_lo[i]), float(x_up[i])
        # Compute z = x_i · inv_rms bounds explicitly. We cannot read z.LB/z.UB
        # right after addVar() — Gurobi defers attribute access until model.update().
        z_lb = min(a_l*inv_rms_lo, a_l*inv_rms_up,
                   a_u*inv_rms_lo, a_u*inv_rms_up)
        z_ub = max(a_l*inv_rms_lo, a_l*inv_rms_up,
                   a_u*inv_rms_lo, a_u*inv_rms_up)
        if a_l == a_u:
            # x_i constant ⇒ z = x_i · inv_rms is linear, no McCormick needed
            z = milp.addVar(lb=z_lb, ub=z_ub, name=f'{prefix}xinv_{i}')
            milp.addConstr(z == a_l * inv_rms, name=f'{prefix}xinv_{i}_def')
        else:
            z = add_mccormick_bilinear(
                milp, x_vars[i], inv_rms,
                a_l, a_u, inv_rms_lo, inv_rms_up,
                prefix=f'{prefix}xinv_{i}_',
            )
        # y_i = gamma_i * z (linear)
        g = float(gamma[i])
        y_lo = g * z_lb if g >= 0 else g * z_ub
        y_up = g * z_ub if g >= 0 else g * z_lb
        y = milp.addVar(lb=y_lo, ub=y_up, name=f'{prefix}y_{i}')
        milp.addConstr(y == g * z, name=f'{prefix}y_def_{i}')
        y_vars.append(y)

    info = dict(
        D            = D,
        n_pieces     = n_pieces,
        mean_xsq_lo  = mean_xsq_lo,
        mean_xsq_up  = mean_xsq_up,
        inv_rms_lo   = inv_rms_lo,
        inv_rms_up   = inv_rms_up,
        n_aux_vars   = len(xsq_vars) + 2,
    )
    return y_vars, info

In [7]:
# ── PyTorch reference RMSNorm (matches notebook 09 exactly) ──────────────
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.dim = dim; self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return self.weight * (x / rms)

def true_rmsnorm(x_np, gamma, eps_rms):
    x = torch.from_numpy(np.asarray(x_np, dtype=np.float64)).unsqueeze(0)  # (1, D)
    layer = RMSNorm(len(gamma), eps=eps_rms).double()
    layer.weight.data = torch.from_numpy(np.asarray(gamma, dtype=np.float64))
    with torch.no_grad():
        y = layer(x).squeeze(0).numpy()
    return y

In [8]:
# ── End-to-end soundness test ────────────────────────────────────────────
def milp_certified_range(x_lo, x_up, gamma, eps_rms,
                         n_pieces=8, time_limit=20.0):
    """
    Build a MILP encoding RMSNorm on the input box, then for each output
    coordinate solve min y_i and max y_i.  Returns (y_min, y_max), both
    arrays of length D.
    """
    if not HAS_GUROBI:
        return None, None
    D = len(x_lo)
    m = gp.Model('rmsnorm_test')
    m.setParam('OutputFlag', 0)
    m.setParam('TimeLimit', time_limit)
    x_vars = [m.addVar(lb=float(x_lo[i]), ub=float(x_up[i]), name=f'x_{i}')
              for i in range(D)]
    y_vars, info = encode_rmsnorm(m, x_vars, x_lo, x_up, gamma, eps_rms,
                                   n_pieces=n_pieces)
    m.update()

    y_min = np.full(D, np.nan); y_max = np.full(D, np.nan)
    for i in range(D):
        m.setObjective(y_vars[i], GRB.MINIMIZE); m.optimize()
        if m.Status not in (GRB.OPTIMAL, GRB.SUBOPTIMAL):
            raise RuntimeError(f'min y_{i} not solved: status={m.Status}')
        y_min[i] = y_vars[i].X
        m.setObjective(y_vars[i], GRB.MAXIMIZE); m.optimize()
        if m.Status not in (GRB.OPTIMAL, GRB.SUBOPTIMAL):
            raise RuntimeError(f'max y_{i} not solved: status={m.Status}')
        y_max[i] = y_vars[i].X
    return y_min, y_max, info


def soundness_test(D: int, x_center: np.ndarray, eps_in: float,
                   gamma: np.ndarray, eps_rms: float,
                   n_pieces: int = 8, K_pins: int = 16,
                   tol: float = 1e-5) -> dict:
    """
    Build the MILP-certified [y_min, y_max] for the input box of radius eps_in
    around x_center, then verify the bracket holds for K_pins random points.
    """
    x_lo = x_center - eps_in
    x_up = x_center + eps_in
    t0 = time.time()
    y_min, y_max, info = milp_certified_range(x_lo, x_up, gamma, eps_rms,
                                              n_pieces=n_pieces)
    solve_t = time.time() - t0

    max_lo_viol = 0.0   # how much true_y under-shot y_min (>0 means UNSOUND)
    max_up_viol = 0.0   # how much true_y over-shot y_max  (>0 means UNSOUND)
    sample_rng = np.random.default_rng(2024)
    for _ in range(K_pins):
        x_pin = x_lo + sample_rng.random(D) * (x_up - x_lo)
        y_true = true_rmsnorm(x_pin, gamma, eps_rms)
        max_lo_viol = max(max_lo_viol, float((y_min - y_true).max()))
        max_up_viol = max(max_up_viol, float((y_true - y_max).max()))
    # Centre point too
    y_true_c = true_rmsnorm(x_center, gamma, eps_rms)
    max_lo_viol = max(max_lo_viol, float((y_min - y_true_c).max()))
    max_up_viol = max(max_up_viol, float((y_true_c - y_max).max()))

    return dict(
        D            = D,
        eps_in       = eps_in,
        n_pieces     = n_pieces,
        solve_time_s = solve_t,
        mean_width   = float((y_max - y_min).mean()),
        max_width    = float((y_max - y_min).max()),
        max_lo_viol  = max_lo_viol,
        max_up_viol  = max_up_viol,
        sound        = (max_lo_viol < tol and max_up_viol < tol),
        info         = info,
    )

In [9]:
# ── Test configurations ──────────────────────────────────────────────────
# Choose D, x_center, eps_in to keep mean(x²) bounded away from 0 so the
# 1/sqrt PWL has finite slope.

EPS_RMS = 1e-6  # matches ViT-Tiny config (notebook 09)

def make_cfg(D, seed=0, scale=0.7, eps_in=0.05):
    g = np.random.default_rng(seed)
    # x_center sampled uniform in [-scale, scale], shifted to ensure mean(x²) > 0.05
    x_center = g.uniform(-scale, scale, size=D)
    # bump magnitudes if mean(x²) too small
    msq = (x_center**2).mean()
    if msq < 0.10:
        x_center = x_center * math.sqrt(0.15 / max(msq, 1e-6))
    gamma = g.uniform(0.5, 1.5, size=D)
    return x_center, gamma, eps_in

CONFIGS = [
    # (label, D, seed, eps_in, n_pieces)
    ('D=4  small box   ', 4,  0, 0.02, 8),
    ('D=4  medium box  ', 4,  1, 0.10, 8),
    ('D=8  small box   ', 8,  2, 0.02, 8),
    ('D=8  medium box  ', 8,  3, 0.05, 8),
    ('D=16 small box   ', 16, 4, 0.02, 8),
    ('D=16 medium box  ', 16, 5, 0.05, 8),
    ('D=32 small box   ', 32, 6, 0.02, 8),
    # n_pieces ablation
    ('D=8  n_pieces=4  ', 8,  7, 0.05, 4),
    ('D=8  n_pieces=16 ', 8,  8, 0.05, 16),
]

if not HAS_GUROBI:
    print('Gurobi unavailable — cannot run MILP soundness tests.')
    results = []
else:
    results = []
    print(f'{"label":22s} {"solve_s":>8s} {"max_w":>10s} {"mean_w":>10s} '
          f'{"lo_viol":>10s} {"up_viol":>10s}  status')
    print('─' * 88)
    for label, D, seed, eps_in, npc in CONFIGS:
        x_center, gamma, eps_in_use = make_cfg(D, seed=seed, eps_in=eps_in)
        rep = soundness_test(D, x_center, eps_in_use, gamma, EPS_RMS,
                             n_pieces=npc, K_pins=24)
        rep['label'] = label
        results.append(rep)
        status = 'SOUND' if rep['sound'] else 'UNSOUND ✗'
        print(f'{label:22s} {rep["solve_time_s"]:8.2f} {rep["max_width"]:10.3e} '
              f'{rep["mean_width"]:10.3e} {rep["max_lo_viol"]:10.2e} '
              f'{rep["max_up_viol"]:10.2e}  {status}')

    n_unsound = sum(1 for r in results if not r['sound'])
    print()
    print(f'Total: {len(results)} configs, {len(results)-n_unsound} sound, '
          f'{n_unsound} unsound')
    assert n_unsound == 0, f'{n_unsound} configs failed soundness'

label                   solve_s      max_w     mean_w    lo_viol    up_viol  status
────────────────────────────────────────────────────────────────────────────────────────
Restricted license - for non-production use only - expires 2027-11-29
D=4  small box             0.15  1.549e-01  1.340e-01   0.00e+00   0.00e+00  SOUND
D=4  medium box            0.15  7.082e-01  5.052e-01   0.00e+00   0.00e+00  SOUND
D=8  small box             0.52  1.984e-01  1.508e-01   0.00e+00   0.00e+00  SOUND
D=8  medium box            0.53  6.172e-01  3.977e-01   0.00e+00   0.00e+00  SOUND
D=16 small box             3.96  2.702e-01  1.566e-01   0.00e+00   0.00e+00  SOUND
D=16 medium box            2.56  6.133e-01  3.491e-01   0.00e+00   0.00e+00  SOUND
D=32 small box            15.44  3.260e-01  1.726e-01   0.00e+00   0.00e+00  SOUND
D=8  n_pieces=4            0.30  4.087e-01  3.414e-01   0.00e+00   0.00e+00  SOUND
D=8  n_pieces=16           1.25  4.546e-01  3.326e-01   0.00e+00   0.00e+00  SOUND

Total: 9 

In [10]:
# ── Save report ──────────────────────────────────────────────────────────
out_dir = Path('results/vit_p2'); out_dir.mkdir(parents=True, exist_ok=True)
report = dict(
    eps_rms = EPS_RMS,
    configs = [{k: v for k, v in r.items() if k != 'info'} for r in results],
)
report_path = out_dir / 'milp_rmsnorm_report.json'
report_path.write_text(json.dumps(report, indent=2, default=float))
print(f'Saved → {report_path}')

# Mirror to Drive (Colab only)
try:
    drive_out = Path('/content/drive/My Drive/thesis-formal-verification/results/vit_p2')
    drive_out.mkdir(parents=True, exist_ok=True)
    (drive_out / 'milp_rmsnorm_report.json').write_text(report_path.read_text())
    print(f'Saved drive copy → {drive_out / "milp_rmsnorm_report.json"}')
except Exception as e:
    print(f'(skipping drive mirror: {e})')

print('\nPhase 4.2 complete: RMSNorm MILP encoding verified sound.')
print('Next: Phase 4.3 (MILP for self-attention; notebook 16).')

Saved → results/vit_p2/milp_rmsnorm_report.json
Saved drive copy → /content/drive/My Drive/thesis-formal-verification/results/vit_p2/milp_rmsnorm_report.json

Phase 4.2 complete: RMSNorm MILP encoding verified sound.
Next: Phase 4.3 (MILP for self-attention; notebook 16).
